# ODE Methods: The Beautiful Failure: original computations


In [ ]:
#| label: ode-methods-cell-1
import numpy as np
import matplotlib.pyplot as plt

# Consistent plotting style for slides
plt.rcParams.update({
    'figure.figsize': (10, 3),
    'font.size': 14,
    'lines.linewidth': 2.5,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'figure.dpi': 100,
    'figure.facecolor': 'white',
})


In [ ]:
#| label: ode-methods-cell-2
def f_decay(t, y):
    """dy/dt = -y (exponential decay)."""
    return -y

# Euler integration
h, t_end = 0.5, 5.0
t_vals = [0.0]
y_vals = [1.0]
while t_vals[-1] < t_end - 1e-12:
    y_new = y_vals[-1] + h * f_decay(t_vals[-1], y_vals[-1])
    t_vals.append(t_vals[-1] + h)
    y_vals.append(y_new)


In [ ]:
#| label: ode-methods-cell-3
t_exact = np.linspace(0, t_end, 200)
fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(t_exact, np.exp(-t_exact), 'k-', lw=2, label='Exact: $e^{-t}$')
ax.plot(t_vals, y_vals, 'ro-', label=f'Euler ($h = {h}$)', markersize=8)
ax.set(xlabel='Time', ylabel='$y(t)$',
       title='Exponential Decay: Euler Works... Okay')
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()


In [ ]:
#| label: ode-methods-cell-4
def f_sho(t, y):
    """Harmonic oscillator: y = [x, v], dy/dt = [v, -omega^2 * x]."""
    x, v = y[0], y[1]
    omega_sq = 1.0  # omega = 1
    return np.array([v, -omega_sq * x])

def euler_integrate(f, y0, t_span, h):
    """Euler integration returning arrays of (t, y)."""
    t_vals = [t_span[0]]
    y_vals = [np.array(y0, dtype=float)]
    while t_vals[-1] < t_span[1] - 1e-12:
        t, y = t_vals[-1], y_vals[-1]
        y_new = y + h * f(t, y)
        t_vals.append(t + h)
        y_vals.append(y_new)
    return np.array(t_vals), np.array(y_vals)

y0_sho = [1.0, 0.0]  # x(0) = 1, v(0) = 0


In [ ]:
#| label: ode-methods-cell-5
h = 0.1
t_end = 20 * np.pi  # 10 full periods
t_euler, y_euler = euler_integrate(f_sho, y0_sho, [0, t_end], h)
t_exact = np.linspace(0, t_end, 2000)


In [ ]:
#| label: ode-methods-cell-6
h = 0.1; t_end = 20 * np.pi  # 10 periods
t_euler, y_euler = euler_integrate(f_sho, y0_sho, [0, t_end], h)

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(np.linspace(0, t_end, 2000), np.cos(np.linspace(0, t_end, 2000)),
        'k-', alpha=0.5, lw=1.5, label='Exact')
ax.plot(t_euler, y_euler[:, 0], 'r-', lw=1, label=f'Euler ($h={h}$)')
ax.set(xlabel='Time', ylabel='$x(t)$')
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
#| label: ode-methods-cell-7
fig, ax = plt.subplots(figsize=(5, 4))
theta = np.linspace(0, 2 * np.pi, 200)
ax.plot(np.cos(theta), -np.sin(theta), 'k--', lw=2, label='Exact (circle)')
ax.plot(y_euler[:, 0], y_euler[:, 1], 'r-', lw=0.5, alpha=0.7,
        label='Euler (spiral)')
ax.set(xlabel='Position $x$', ylabel='Velocity $v$',
       title='Phase Space: Euler Spirals Outward')
ax.set_aspect('equal')
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
#| label: ode-methods-cell-8
# Total energy: E = (1/2)(v^2 + omega^2 * x^2). For omega=1: E = (x^2 + v^2)/2
E_euler = 0.5 * (y_euler[:, 0]**2 + y_euler[:, 1]**2)

fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(t_euler, E_euler, 'r-', lw=2, label='Euler')
ax.axhline(0.5, color='k', ls='--', lw=1.5, label='Exact ($E = 0.5$)')
ax.set(xlabel='Time', ylabel='Total Energy $E$',
       title='Energy Conservation: Euler FAILS')
ax.legend()
ax.set_ylim(0, min(E_euler[-1] * 1.1, E_euler[-1] + 50))
plt.tight_layout()
plt.show()

print(f"Energy at start: {E_euler[0]:.4f}")
print(f"Energy at end:   {E_euler[-1]:.1f}")
print(f"Energy grew by:  {(E_euler[-1] / E_euler[0] - 1) * 100:.0f}%")


In [ ]:
#| label: ode-methods-cell-9
def rk2_step(f, t, y, h):
    """One step of RK2 (midpoint method): equations become code."""
    k1 = f(t, y)                          # slope at start
    k2 = f(t + h/2, y + h/2 * k1)        # slope at midpoint
    return y + h * k2                      # full step with better slope


In [ ]:
#| label: ode-methods-cell-10
def rk2_integrate(f, y0, t_span, h):
    """RK2 integration — same structure as euler_integrate."""
    t_vals = [t_span[0]]
    y_vals = [np.array(y0, dtype=float)]
    while t_vals[-1] < t_span[1] - 1e-12:
        t, y = t_vals[-1], y_vals[-1]
        y_new = rk2_step(f, t, y, h)     # <-- only this line changes!
        t_vals.append(t + h)
        y_vals.append(y_new)
    return np.array(t_vals), np.array(y_vals)


In [ ]:
#| label: ode-methods-cell-11
t_rk2, y_rk2 = rk2_integrate(f_sho, y0_sho, [0, t_end], h)

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(t_exact, np.cos(t_exact), 'k-', alpha=0.4, lw=1.5, label='Exact')
ax.plot(t_euler, y_euler[:, 0], 'r-', lw=0.8, alpha=0.5, label='Euler')
ax.plot(t_rk2, y_rk2[:, 0], 'b-', lw=1.5, label='RK2')
ax.set(xlabel='Time', ylabel='$x(t)$',
       title='RK2 vs Euler: Night and Day')
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
#| label: ode-methods-cell-12
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.5))
theta = np.linspace(0, 2 * np.pi, 200)

ax1.plot(np.cos(theta), -np.sin(theta), 'k--', lw=2)
ax1.plot(y_euler[:, 0], y_euler[:, 1], 'r-', lw=0.3, alpha=0.6)
ax1.set(title='Euler: Spiral Out', xlabel='$x$', ylabel='$v$')
ax1.set_aspect('equal')

ax2.plot(np.cos(theta), -np.sin(theta), 'k--', lw=2)
ax2.plot(y_rk2[:, 0], y_rk2[:, 1], 'b-', lw=0.5, alpha=0.7)
ax2.set(title='RK2: Much Tighter', xlabel='$x$', ylabel='$v$')
ax2.set_aspect('equal')

plt.tight_layout()
plt.show()


In [ ]:
#| label: ode-methods-cell-13
E_rk2 = 0.5 * (y_rk2[:, 0]**2 + y_rk2[:, 1]**2)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3))

# Left: both on same scale
ax1.plot(t_euler, E_euler, 'r-', lw=1.5, label='Euler')
ax1.plot(t_rk2, E_rk2, 'b-', lw=1.5, label='RK2')
ax1.axhline(0.5, color='k', ls='--', lw=1)
ax1.set(xlabel='Time', ylabel='Energy $E$', title='Full Scale')
ax1.legend()

# Right: zoom on RK2
ax2.plot(t_rk2, E_rk2 - 0.5, 'b-', lw=1.5)
ax2.axhline(0, color='k', ls='--', lw=1)
ax2.set(xlabel='Time', ylabel='$E - E_0$', title='RK2 Energy Error (Zoomed)')

plt.tight_layout()
plt.show()

print(f"Euler final energy error: {abs(E_euler[-1] - 0.5):.1f}")
print(f"RK2 final energy error:   {abs(E_rk2[-1] - 0.5):.2e}")
print(f"RK2 is {abs(E_euler[-1] - 0.5) / abs(E_rk2[-1] - 0.5):.0f}x better")
